In [ ]:
# INTERAKTÍV TENISZ MECCS PREDIKTOR

import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import pickle
import os
from datetime import datetime

# MODELL BETÖLTÉSE
def load_latest_model():
    """Legfrissebb modell betöltése"""
    models_dir = "models"
    
    if not os.path.exists(models_dir):
        print(f"❌ Models mappa nem található: {models_dir}")
        return None
    
    # Összes .pkl fájl keresése
    model_files = [f for f in os.listdir(models_dir) if f.endswith('.pkl')]
    
    if not model_files:
        print(f"❌ Nincs mentett modell a {models_dir} mappában")
        return None
    
    # Legfrissebb fájl kiválasztása
    latest_file = sorted(model_files)[-1]
    model_path = os.path.join(models_dir, latest_file)
    
    try:
        with open(model_path, 'rb') as f:
            model_data = pickle.load(f)
        print(f"✅ Modell betöltve: {latest_file}")
        return model_data
    except Exception as e:
        print(f"❌ Hiba a modell betöltésekor: {str(e)}")
        return None

# MODELL BETÖLTÉSE
print("Modell betöltése...")
model_data = load_latest_model()

if model_data is None:
    print("❌ Nem sikerült betölteni a modellt!")
else:
    model = model_data['model']
    features = model_data['features']
    feature_names = model_data['feature_names']
    print("✅ Modell sikeresen betöltve!")

# WIDGET-EK LÉTREHOZÁSA
print("\nInteraktív prediktor létrehozása...")

# CSS stílusok
style = {'description_width': '200px'}
layout = widgets.Layout(width='400px')

# 1. JÁTÉKOS NEVEK
player1_name = widgets.Text(
    value='Novak Djokovic',
    placeholder='Player 1 neve',
    description='🎾 Player 1:',
    style=style,
    layout=layout
)

player2_name = widgets.Text(
    value='Rafael Nadal', 
    placeholder='Player 2 neve',
    description='🎾 Player 2:',
    style=style,
    layout=layout
)

# 2. ODDS ÉRTÉKEK
player1_odds = widgets.FloatSlider(
    value=2.5,
    min=1.1,
    max=10.0,
    step=0.1,
    description='📊 Player 1 Odds:',
    style=style,
    layout=layout,
    readout_format='.2f'
)

player2_odds = widgets.FloatSlider(
    value=1.8,
    min=1.1,
    max=10.0, 
    step=0.1,
    description='📊 Player 2 Odds:',
    style=style,
    layout=layout,
    readout_format='.2f'
)

# 3. RANGSOR ÉRTÉKEK
player1_rank = widgets.IntSlider(
    value=2,
    min=1,
    max=200,
    step=1,
    description='🏆 Player 1 Ranking:',
    style=style,
    layout=layout
)

player2_rank = widgets.IntSlider(
    value=1,
    min=1,
    max=200,
    step=1,
    description='🏆 Player 2 Ranking:',
    style=style,
    layout=layout
)

# 4. HEAD-TO-HEAD STATISZTIKÁK
h2h_matches = widgets.IntSlider(
    value=5,
    min=0,
    max=50,
    step=1,
    description='🤝 H2H meccsek száma:',
    style=style,
    layout=layout
)

h2h_p1_wins = widgets.IntSlider(
    value=2,
    min=0,
    max=50,
    step=1,
    description='🏅 Player 1 H2H győzelmek:',
    style=style,
    layout=layout
)

h2h_last_winner = widgets.Dropdown(
    options=[('Player 1', 1), ('Player 2', 0), ('Nincs adat', 0.5)],
    value=1,
    description='👑 Utolsó H2H győztes:',
    style=style,
    layout=layout
)

# 5. FELÜLET TÍPUSA (informatív)
surface_type = widgets.Dropdown(
    options=[('Hard Court', 'Hard'), ('Clay Court', 'Clay'), ('Grass Court', 'Grass')],
    value='Hard',
    description='🎾 Felület típusa:',
    style=style,
    layout=layout
)

# PREDIKCIÓ GOMB
predict_button = widgets.Button(
    description='🔮 PREDIKCIÓ FUTTATÁSA',
    button_style='success',
    layout=widgets.Layout(width='300px', height='50px')
)

# EREDMÉNYEK MEGJELENÍTÉSE
output = widgets.Output()

# PREDIKCIÓ FÜGGVÉNY
def make_prediction(b):
    """Predikció készítése a widget értékek alapján"""
    with output:
        clear_output()
        
        try:
            # Input értékek kinyerése
            p1_odds = player1_odds.value
            p2_odds = player2_odds.value
            p1_rank = player1_rank.value
            p2_rank = player2_rank.value
            h2h_total = h2h_matches.value
            h2h_p1_wins_val = h2h_p1_wins.value
            h2h_last = h2h_last_winner.value
            
            # Validáció
            if h2h_p1_wins_val > h2h_total:
                print("❌ Hiba: Player 1 H2H győzelmek nem lehet több mint az összes H2H meccs!")
                return
            
            # Feature-ök számítása
            odds_diff = p2_odds - p1_odds  # Player2 odds - Player1 odds
            implied_prob1 = 1 / p1_odds   # Player1 implicit valószínűség
            implied_prob2 = 1 / p2_odds   # Player2 implicit valószínűség
            rank_diff = p2_rank - p1_rank  # Player2 rank - Player1 rank
            h2h_p1_winrate = h2h_p1_wins_val / h2h_total if h2h_total > 0 else 0.5
            
            # Predikciós adatok
            prediction_data = pd.DataFrame({
                'Odds_Diff': [odds_diff],
                'Implied_Prob1': [implied_prob1], 
                'Implied_Prob2': [implied_prob2],
                'H2H_P1_WinRate': [h2h_p1_winrate],
                'H2H_LastWinnerP1': [h2h_last],
                'Rank_Diff': [rank_diff]
            })
            
            # Predikció
            win_probability = model.predict_proba(prediction_data)[0, 1]
            
            # EREDMÉNYEK MEGJELENÍTÉSE
            print("=" * 60)
            print("🎾 TENISZ MECCS PREDIKCIÓ")
            print("=" * 60)
            print()
            
            # Játékos információk
            print(f"🏆 {player1_name.value} (#{p1_rank}) vs {player2_name.value} (#{p2_rank})")
            print(f"🎾 Felület: {surface_type.value}")
            print()
            
            # Odds információk
            print("📊 ODDS INFORMÁCIÓK:")
            print(f"   • {player1_name.value}: {p1_odds:.2f} (Implicit: {implied_prob1:.3f})")
            print(f"   • {player2_name.value}: {p2_odds:.2f} (Implicit: {implied_prob2:.3f})")
            favorit = player1_name.value if p1_odds < p2_odds else player2_name.value
            print(f"   • Favorit: {favorit}")
            print()
            
            # H2H információk
            if h2h_total > 0:
                h2h_p2_wins = h2h_total - h2h_p1_wins_val
                print("🤝 HEAD-TO-HEAD:")
                print(f"   • Összes meccs: {h2h_total}")
                print(f"   • {player1_name.value}: {h2h_p1_wins_val} győzelem ({h2h_p1_winrate:.1%})")
                print(f"   • {player2_name.value}: {h2h_p2_wins} győzelem ({(1-h2h_p1_winrate):.1%})")
                last_winner_name = player1_name.value if h2h_last == 1 else player2_name.value if h2h_last == 0 else "Nincs adat"
                print(f"   • Utolsó győztes: {last_winner_name}")
            else:
                print("🤝 HEAD-TO-HEAD: Nincs korábbi meccs")
            print()
            
            # PREDIKCIÓ EREDMÉNYE
            print("🔮 PREDIKCIÓ EREDMÉNYE:")
            print("-" * 40)
            
            # Győzelmi valószínűségek
            p1_prob = win_probability
            p2_prob = 1 - win_probability
            
            print(f"🏅 {player1_name.value} győzelmi esély: {p1_prob:.1%}")
            print(f"🏅 {player2_name.value} győzelmi esély: {p2_prob:.1%}")
            print()
            
            # Előrejelzett győztes
            predicted_winner = player1_name.value if p1_prob > p2_prob else player2_name.value
            confidence = max(p1_prob, p2_prob)
            
            if confidence > 0.7:
                confidence_level = "Magas bizonyosság 🔥"
                confidence_color = "🟢"
            elif confidence > 0.6:
                confidence_level = "Közepes bizonyosság ⚡"
                confidence_color = "🟡"
            else:
                confidence_level = "Alacsony bizonyosság ❓"
                confidence_color = "🔴"
            
            print(f"{confidence_color} ELŐREJELZETT GYŐZTES: {predicted_winner}")
            print(f"   Bizonyossági szint: {confidence:.1%} ({confidence_level})")
            print()
            
            # VALUE BETTING ELEMZÉS
            print("💰 VALUE BETTING ELEMZÉS:")
            print("-" * 40)
            
            # Player 1 value
            p1_fair_odds = 1 / p1_prob
            p1_value = (p1_fair_odds - p1_odds) / p1_odds * 100
            p1_kelly = max(0, (p1_prob * (p1_odds - 1) - (1 - p1_prob)) / (p1_odds - 1))
            
            # Player 2 value  
            p2_fair_odds = 1 / p2_prob
            p2_value = (p2_fair_odds - p2_odds) / p2_odds * 100
            p2_kelly = max(0, (p2_prob * (p2_odds - 1) - (1 - p2_prob)) / (p2_odds - 1))
            
            print(f"📈 {player1_name.value}:")
            print(f"   • Fair odds: {p1_fair_odds:.2f}")
            print(f"   • Value: {p1_value:+.1f}%")
            if p1_value > 0:
                print(f"   • Kelly%: {p1_kelly:.1%} 💚")
            else:
                print(f"   • Nincs value ❌")
            
            print(f"📈 {player2_name.value}:")
            print(f"   • Fair odds: {p2_fair_odds:.2f}")  
            print(f"   • Value: {p2_value:+.1f}%")
            if p2_value > 0:
                print(f"   • Kelly%: {p2_kelly:.1%} 💚")
            else:
                print(f"   • Nincs value ❌")
            
            print()
            
            # BETTING AJÁNLÁS
            print("🎯 BETTING AJÁNLÁS:")
            print("-" * 40)
            
            best_value = max(p1_value, p2_value)
            
            if best_value > 5:
                if p1_value > p2_value:
                    print(f"🟢 ERŐS AJÁNLÁS: Fogadj {player1_name.value}-re!")
                    print(f"   Javasolt tét: {min(p1_kelly * 100, 5):.1f}% a bankroll-ból")
                else:
                    print(f"🟢 ERŐS AJÁNLÁS: Fogadj {player2_name.value}-re!")
                    print(f"   Javasolt tét: {min(p2_kelly * 100, 5):.1f}% a bankroll-ból")
            elif best_value > 2:
                winner = player1_name.value if p1_value > p2_value else player2_name.value
                kelly_val = p1_kelly if p1_value > p2_value else p2_kelly
                print(f"🟡 GYENGE AJÁNLÁS: Fontold meg {winner}-t")
                print(f"   Javasolt tét: {min(kelly_val * 100, 2):.1f}% a bankroll-ból")
            else:
                print("🔴 NINCS VALUE: Ne fogadj erre a meccsre")
            
            print()
            print("⚠️  Kockázati figyelmeztetés: A sportfogadás kockázatos!")
            print("=" * 60)
            
        except Exception as e:
            print(f"❌ Hiba a predikció során: {str(e)}")
            import traceback
            traceback.print_exc()

# Event handler hozzáadása
predict_button.on_click(make_prediction)

# H2H wins validálás
def validate_h2h_wins(change):
    """H2H wins validálása"""
    if h2h_p1_wins.value > h2h_matches.value:
        h2h_p1_wins.value = h2h_matches.value

h2h_matches.observe(validate_h2h_wins, names='value')
h2h_p1_wins.observe(validate_h2h_wins, names='value')

# LAYOUT ÖSSZEÁLLÍTÁSA
print("✅ Interaktív prediktor kész!")
print("\n" + "="*60)
print("🎾 TENISZ MECCS PREDIKTOR")
print("="*60)

# Widget-ek megjelenítése
display(widgets.HTML("<h2>🎾 Játékos Információk</h2>"))
display(widgets.VBox([player1_name, player2_name, surface_type]))

display(widgets.HTML("<h2>📊 Odds & Rangsor</h2>"))
display(widgets.VBox([player1_odds, player2_odds, player1_rank, player2_rank]))

display(widgets.HTML("<h2>🤝 Head-to-Head Statisztikák</h2>"))
display(widgets.VBox([h2h_matches, h2h_p1_wins, h2h_last_winner]))

display(widgets.HTML("<h2>🔮 Predikció</h2>"))
display(predict_button)
display(output)

# FEATURE MAGYARÁZAT
display(widgets.HTML("""
<h3>📋 Feature Magyarázatok:</h3>
<ul>
<li><b>Odds_Diff:</b> Player2 odds - Player1 odds (pozitív = Player1 favorit)</li>
<li><b>Implied_Prob1/2:</b> Implicit valószínűség az odds alapján (1/odds)</li>
<li><b>Rank_Diff:</b> Player2 rangsor - Player1 rangsor (pozitív = Player1 jobb)</li>
<li><b>H2H_P1_WinRate:</b> Player1 győzelmi aránya korábbi meccsekben</li>
<li><b>H2H_LastWinnerP1:</b> Utolsó H2H meccs győztese</li>
</ul>
<p><i>💡 Tipp: Állítsd be a real-time odds értékeket a pontosabb predikció érdekében!</i></p>
"""))